In [61]:
import torch
import torchvision.models as models
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from PIL import Image
import os
from sklearn.metrics import roc_auc_score
import torch.nn.functional as F
import numpy as np
import torch.nn as nn
import torch.optim as optim
import json
from tqdm.notebook import tqdm
import gc
import timm

In [62]:
gc.collect()
# Clear PyTorch's resident memory
torch.cuda.empty_cache()

Face verification data loading for AUC and ROC calculation

In [63]:
class FaceVerificationDataset(torch.utils.data.Dataset):
    def __init__(self, txt_file, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.pairs = []

        with open(txt_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 3:
                    self.pairs.append((parts[0], parts[1], int(parts[2])))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img1_path, img2_path, label = self.pairs[idx]

        img1_full_path = os.path.join(self.root_dir, img1_path)
        img2_full_path = os.path.join(self.root_dir, img2_path)

        img1 = Image.open(img1_full_path).convert('RGB')
        img2 = Image.open(img2_full_path).convert('RGB')

        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        return img1, img2, torch.tensor(label, dtype=torch.float32)

Data loading and augmentation:

In [64]:

transformation = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), 
    transforms.RandomRotation(degrees=10, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [65]:
train_dataset = ImageFolder(root='../data/classification_data/train_data', transform=transformation)
train_loader = DataLoader(
    train_dataset, 
    batch_size=64, 
    shuffle=True,    
    num_workers=4,   
    pin_memory=True   
)

In [66]:
class_val_dataset = ImageFolder(
    root='../data/classification_data/val_data', 
    transform=transformation,

)
class_val_loader = DataLoader(
    class_val_dataset, 
    batch_size=64, 
    shuffle=False, 
    num_workers=4
)

verfication loading

In [67]:
verification_root = '../data' 

verification_dataset = FaceVerificationDataset(
    txt_file='../data/verification_pairs_val.txt', 
    root_dir=verification_root, 
    transform= transformation
)

verification_loader = torch.utils.data.DataLoader(verification_dataset, batch_size=32, shuffle=False)

Calculating ROC and AUC

In [68]:
def validate_verification_auc(model, val_loader, device):
    model.eval()
    all_labels = []
    all_cosine_scores = []
    all_euclidean_distances = []

    with torch.no_grad():
        for img1, img2, labels in val_loader:
            img1, img2 = img1.to(device), img2.to(device)

            feat1 = model.features(img1)
            feat1 = model.avgpool(feat1).flatten(1)
            
            feat2 = model.features(img2)
            feat2 = model.avgpool(feat2).flatten(1)

            cos_sim = F.cosine_similarity(feat1, feat2)
            euc_dist = torch.cdist(feat1.unsqueeze(1), feat2.unsqueeze(1)).squeeze(2).squeeze(1)
            
            all_cosine_scores.extend(cos_sim.cpu().numpy())
            all_euclidean_distances.extend(euc_dist.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            

    auc_cosine = roc_auc_score(all_labels, all_cosine_scores)
    auc_euclidean = roc_auc_score(all_labels, -np.array(all_euclidean_distances))

    return auc_cosine, auc_euclidean

In [69]:
class MobileViT(nn.Module):
    def __init__(self, num_classes=1):
        super(MobileViT, self).__init__()
        
        self.base_model = timm.create_model('mobilevit_xs.cvnets_in1k', pretrained=True)
    
        self.avgpool = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(1)
        )
    

        self.classifier = nn.Sequential(
            nn.Linear(384, num_classes)
        )

    def features(self, x):
            return self.base_model.forward_features(x)

    def forward(self, x):

        x = self.features(x)
        x = self.avgpool(x)
        x = self.classifier(x)
        return x

In [70]:
num_classes = len(train_dataset.classes)
model = MobileViT(num_classes)

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")


Layer: base_model.stem.conv.weight | Size: torch.Size([16, 3, 3, 3])
Layer: base_model.stem.bn.weight | Size: torch.Size([16])
Layer: base_model.stem.bn.bias | Size: torch.Size([16])
Layer: base_model.stages.0.0.conv1_1x1.conv.weight | Size: torch.Size([64, 16, 1, 1])
Layer: base_model.stages.0.0.conv1_1x1.bn.weight | Size: torch.Size([64])
Layer: base_model.stages.0.0.conv1_1x1.bn.bias | Size: torch.Size([64])
Layer: base_model.stages.0.0.conv2_kxk.conv.weight | Size: torch.Size([64, 1, 3, 3])
Layer: base_model.stages.0.0.conv2_kxk.bn.weight | Size: torch.Size([64])
Layer: base_model.stages.0.0.conv2_kxk.bn.bias | Size: torch.Size([64])
Layer: base_model.stages.0.0.conv3_1x1.conv.weight | Size: torch.Size([32, 64, 1, 1])
Layer: base_model.stages.0.0.conv3_1x1.bn.weight | Size: torch.Size([32])
Layer: base_model.stages.0.0.conv3_1x1.bn.bias | Size: torch.Size([32])
Layer: base_model.stages.1.0.conv1_1x1.conv.weight | Size: torch.Size([128, 32, 1, 1])
Layer: base_model.stages.1.0.conv1_

In [71]:
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()

In [72]:
print(f"Is CUDA available? {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Device Count: {torch.cuda.device_count()}")

Is CUDA available? True
CUDA version: 13.0
Device Count: 1


In [73]:
num_epochs = 30
best_auc = 0.0
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss':[],
    'val_acc':[],
    'cos_auc': [],
    'euc_auc': []
}

scaler = torch.amp.GradScaler()

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_preds = 0
    total_preds = 0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}]", leave=True)

    # --- TRAINING PHASE ---
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type=device.type):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total_preds += labels.size(0)
        correct_preds += (predicted == labels).sum().item()
        loop.set_postfix(loss=loss.item(), acc=correct_preds/total_preds)

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in class_val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            
            # Calculate loss and accuracy
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_preds / total_preds
    epoch_val_loss = val_loss / len(class_val_dataset)
    epoch_val_acc = val_correct / val_total


    cos_auc, euc_auc = validate_verification_auc(model, verification_loader, device)

    history['train_loss'].append(epoch_loss)
    history['train_acc'].append(epoch_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    history['cos_auc'].append(cos_auc)
    history['euc_auc'].append(euc_auc)

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f}")
    print(f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")
    print(f"Cosine AUC: {cos_auc:.4f} | Euclidean AUC: {euc_auc:.4f}")
    print("-" * 30)

    if cos_auc > best_auc:
        best_auc = cos_auc
        torch.save(model.state_dict(), 'best_supervised_model_ViT.pth')
        print("Model saved based on Cosine AUC!")

    scheduler.step(cos_auc)


# Save history to a JSON file
with open('training_history_supervised_ViT.json', 'w+') as f:
    json.dump(history, f)

Epoch [1/30]:   0%|          | 0/5948 [00:00<?, ?it/s]

Epoch [1/30]
Train Loss: 6.7294 | Train Acc: 0.0251
Val Loss: 5.6738 | Val Acc: 0.0532
Cosine AUC: 0.8748 | Euclidean AUC: 0.8536
------------------------------
Model saved based on Cosine AUC!


Epoch [2/30]:   0%|          | 0/5948 [00:00<?, ?it/s]

Epoch [2/30]
Train Loss: 4.8711 | Train Acc: 0.1332
Val Loss: 4.3389 | Val Acc: 0.1751
Cosine AUC: 0.8984 | Euclidean AUC: 0.8885
------------------------------
Model saved based on Cosine AUC!


Epoch [3/30]:   0%|          | 0/5948 [00:00<?, ?it/s]

...

Epoch [28/30]
Train Loss: 0.8762 | Train Acc: 0.8013
Val Loss: 1.4295 | Val Acc: 0.6976
Cosine AUC: 0.9128 | Euclidean AUC: 0.8822
------------------------------


Epoch [29/30]:   0%|          | 0/5948 [00:00<?, ?it/s]

Epoch [29/30]
Train Loss: 0.8768 | Train Acc: 0.8012
Val Loss: 1.4247 | Val Acc: 0.6984
Cosine AUC: 0.9140 | Euclidean AUC: 0.8845
------------------------------


Epoch [30/30]:   0%|          | 0/5948 [00:00<?, ?it/s]

Epoch [30/30]
Train Loss: 0.8770 | Train Acc: 0.8010
Val Loss: 1.4342 | Val Acc: 0.6977
Cosine AUC: 0.9131 | Euclidean AUC: 0.8844
------------------------------
